In [2]:
pip install jieba snownlppip install jieba snownlp

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement snownlppip (from versions: none)
ERROR: No matching distribution found for snownlppip


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error, make_scorer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import re
from datetime import datetime
import warnings
from scipy import stats
import chardet

warnings.filterwarnings("ignore")

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

class AdvancedRealEstateProcessor:
    """基于Midterm_codes思路改进的房地产数据处理类"""
    
    def __init__(self):
        self.selected_features = None
        self.feature_stats = {}
        self.imputer = None
        
    def detect_encoding(self, file_path):
        with open(file_path, 'rb') as f:
            result = chardet.detect(f.read())
        return result.get('encoding', 'utf-8')
    
    def load_and_combine_data(self, train_path, test_path, target_col='Price'):
        """加载并合并训练和测试数据，便于统一预处理"""
        try:
            # 检测编码
            train_encoding = self.detect_encoding(train_path)
            test_encoding = self.detect_encoding(test_path)
            
            # 加载数据
            train_df = pd.read_csv(train_path, encoding=train_encoding, low_memory=False)
            test_df = pd.read_csv(test_path, encoding=test_encoding, low_memory=False)
            
            print(f"数据加载成功: 训练集 {train_df.shape}, 测试集 {test_df.shape}")
            
            # 添加标识列
            train_df['is_train'] = 1
            test_df['is_train'] = 0
            
            # 合并数据
            combined_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
            
            return combined_df, train_df, test_df
            
        except Exception as e:
            print(f"数据加载失败: {e}")
            return None, None, None
    
    def remove_data_leakage(self, df):
        """移除数据泄露特征"""
        leakage_patterns = [
            'community', 'historical', 'appraisal', '评估', '均价',
            'avg', 'mean', 'predicted', 'estimated', '小区', '楼盘'
        ]
        leakage_cols = [col for col in df.columns if any(pat in col.lower() for pat in leakage_patterns)]
        if leakage_cols:
            print(f"移除数据泄露特征: {leakage_cols}")
            df = df.drop(columns=leakage_cols, errors='ignore')
        return df
    
    def parse_room_layout(self, layout_series):
        """解析房屋户型，提取室、厅、卫数量"""
        rooms, living_rooms, bathrooms = [], [], []
        
        for layout in layout_series:
            if pd.isna(layout):
                rooms.append(np.nan)
                living_rooms.append(np.nan)
                bathrooms.append(np.nan)
                continue
                
            layout_str = str(layout)
            # 提取数字
            room_match = re.search(r'(\d+)室', layout_str)
            living_match = re.search(r'(\d+)厅', layout_str)  
            bath_match = re.search(r'(\d+)卫', layout_str)
            
            rooms.append(float(room_match.group(1)) if room_match else np.nan)
            living_rooms.append(float(living_match.group(1)) if living_match else np.nan)
            bathrooms.append(float(bath_match.group(1)) if bath_match else np.nan)
            
        return rooms, living_rooms, bathrooms
    
    def parse_floor_info(self, floor_series):
        """解析楼层信息"""
        total_floors, floor_positions = [], []
        
        for floor in floor_series:
            if pd.isna(floor):
                total_floors.append(np.nan)
                floor_positions.append(np.nan)
                continue
                
            floor_str = str(floor)
            # 提取总楼层
            total_match = re.search(r'共(\d+)层', floor_str)
            total_floors.append(float(total_match.group(1)) if total_match else np.nan)
            
            # 楼层位置映射
            if '地下室' in floor_str:
                floor_positions.append(-1)
            elif '底层' in floor_str or '低楼层' in floor_str:
                floor_positions.append(1)
            elif '中楼层' in floor_str:
                floor_positions.append(2)  
            elif '高楼层' in floor_str:
                floor_positions.append(3)
            elif '顶层' in floor_str:
                floor_positions.append(4)
            else:
                floor_positions.append(np.nan)
                
        return total_floors, floor_positions
    
    def parse_building_year(self, year_series):
        """解析建筑年代"""
        building_years = []
        
        for year in year_series:
            if pd.isna(year):
                building_years.append(np.nan)
                continue
                
            year_str = str(year)
            years = re.findall(r'\d{4}', year_str)
            if years:
                # 取平均年份（如果有多个）
                building_years.append(np.mean([float(y) for y in years]))
            else:
                building_years.append(np.nan)
                
        return building_years
    
    def create_advanced_features(self, df):
        """创建高级特征 - 基于Midterm_codes思路但重新实现"""
        df_feat = df.copy()
        
        # 1. 解析房屋户型
        if '房屋户型' in df_feat.columns:
            rooms, living_rooms, bathrooms = self.parse_room_layout(df_feat['房屋户型'])
            df_feat['房间数'] = rooms
            df_feat['客厅数'] = living_rooms  
            df_feat['卫生间数'] = bathrooms
        
        # 2. 解析楼层信息
        if '所在楼层' in df_feat.columns:
            total_floors, floor_positions = self.parse_floor_info(df_feat['所在楼层'])
            df_feat['总楼层'] = total_floors
            df_feat['楼层位置'] = floor_positions
        
        # 3. 解析建筑年代
        if '建筑年代' in df_feat.columns:
            building_years = self.parse_building_year(df_feat['建筑年代'])
            df_feat['建筑年份'] = building_years
            df_feat['房龄'] = 2025 - df_feat['建筑年份']  # 假设当前年份为2025
        
        # 4. 面积处理
        area_columns = [col for col in df_feat.columns if '面积' in col or '建筑面积' in col]
        for col in area_columns:
            if col in df_feat.columns:
                # 提取数字并转换
                area_values = df_feat[col].astype(str).str.replace('㎡', '').str.extract(r'(\d+\.?\d*)')[0]
                df_feat[f'{col}_数值'] = pd.to_numeric(area_values, errors='coerce')
                # 对数转换前处理零值和负值
                valid_areas = df_feat[f'{col}_数值'].fillna(0)
                valid_areas = np.where(valid_areas <= 0, np.nan, valid_areas)
                df_feat[f'{col}_对数'] = np.log1p(valid_areas)
        
        # 5. 朝向特征
        if '房屋朝向' in df_feat.columns:
            df_feat['朝南'] = df_feat['房屋朝向'].str.contains('南', na=False).astype(int)
            df_feat['朝北'] = df_feat['房屋朝向'].str.contains('北', na=False).astype(int) 
            df_feat['朝东'] = df_feat['房屋朝向'].str.contains('东', na=False).astype(int)
            df_feat['朝西'] = df_feat['房屋朝向'].str.contains('西', na=False).astype(int)
        
        # 6. 装修情况映射
        if '装修情况' in df_feat.columns:
            renovation_map = {'毛坯': 0, '简装': 1, '精装': 2}
            df_feat['装修等级'] = df_feat['装修情况'].map(renovation_map).fillna(1)  # 默认简装
        
        # 7. 梯户比例解析
        if '梯户比例' in df_feat.columns:
            # 中文数字映射
            chinese_num_map = {
                "一": "1", "二": "2", "两": "2", "三": "3", "四": "4", 
                "五": "5", "六": "6", "七": "7", "八": "8", "九": "9", "十": "10"
            }
            temp_col = df_feat['梯户比例'].astype(str)
            for char, num in chinese_num_map.items():
                temp_col = temp_col.str.replace(char, num)
            
            df_feat['电梯数'] = temp_col.str.extract(r'(\d+)梯')[0].astype(float)
            df_feat['每层户数'] = temp_col.str.extract(r'(\d+)户')[0].astype(float)
            # 避免除零错误
            df_feat['梯户比'] = df_feat['电梯数'] / df_feat['每层户数'].replace(0, np.nan)
        
        # 8. 日期特征
        date_columns = ['交易时间', '上次交易']
        for col in date_columns:
            if col in df_feat.columns:
                try:
                    df_feat[col] = pd.to_datetime(df_feat[col], errors='coerce')
                    df_feat[f'{col}_年份'] = df_feat[col].dt.year
                    df_feat[f'{col}_月份'] = df_feat[col].dt.month
                except:
                    pass
        
        # 9. 计算持有天数（如果有交易时间）
        if '交易时间' in df_feat.columns and '上次交易' in df_feat.columns:
            df_feat['持有天数'] = (df_feat['交易时间'] - df_feat['上次交易']).dt.days
            df_feat['是否首次交易'] = df_feat['上次交易'].isnull().astype(int)
        
        # 10. 文本特征标志
        text_columns = ['房屋优势', '周边配套', '交通出行', '核心卖点']
        for col in text_columns:
            if col in df_feat.columns:
                df_feat[f'有_{col}'] = df_feat[col].notnull().astype(int)
        
        return df_feat
    
    def handle_missing_values(self, df, target_col='Price'):
        """处理缺失值，只对训练集移除目标缺失样本"""
        df_clean = df.copy()
        
        # 保存 is_train 列
        is_train_col = None
        if 'is_train' in df_clean.columns:
            is_train_col = df_clean['is_train'].copy()
        
        # 找出数值列并计算缺失率
        numeric_cols_initial = df_clean.select_dtypes(include=[np.number]).columns
        missing_rates = df_clean[numeric_cols_initial].isnull().mean()
        high_missing_cols = missing_rates[missing_rates > 0.5].index.tolist()
        
        if high_missing_cols:
            print(f"移除高缺失率特征: {high_missing_cols}")
            df_clean = df_clean.drop(columns=high_missing_cols, errors='ignore')
        
        # 重新计算当前存在的数值列
        numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
        
        # 排除目标列
        if target_col in numeric_cols:
            numeric_cols = [c for c in numeric_cols if c != target_col]
        
        # 用中位数填充现存的数值列的缺失值
        for col in numeric_cols:
            null_count = df_clean[col].isnull().sum()
            if null_count > 0:
                median_val = df_clean[col].median()
                df_clean[col].fillna(median_val, inplace=True)
                print(f"已用中位数填充数值列 {col}：填充 {null_count} 个缺失值（中位数={median_val}）")
        
        # 只对训练集移除目标缺失样本
        if target_col in df_clean.columns:
            train_mask = df_clean['is_train'] == 1
            test_mask = df_clean['is_train'] == 0
            
            before_target = train_mask.sum()
            train_clean = df_clean[train_mask].dropna(subset=[target_col])
            after_target = len(train_clean)
            
            if before_target != after_target:
                print(f"移除训练集目标变量缺失样本: {before_target - after_target} 条, 剩余 {after_target} 条")
            
            # 重新组合数据
            test_clean = df_clean[test_mask]
            df_clean = pd.concat([train_clean, test_clean], axis=0)
        
        # 恢复 is_train 列
        if is_train_col is not None:
            df_clean['is_train'] = is_train_col.reindex(df_clean.index).fillna(0).astype(int)
        
        print("✅ 缺失值处理完成，is_train 列已恢复并对齐。")
        return df_clean

    def remove_outliers(self, df, target_col='Price'):
        """移除异常值，只对训练集进行处理"""
        if df is None:
            return df
    
        df_clean = df.copy()
    
        # 保存 is_train 列
        is_train_col = None
        if 'is_train' in df_clean.columns:
            is_train_col = df_clean['is_train'].copy()
    
        # 分离训练集和测试集
        train_df = df_clean[df_clean['is_train'] == 1].copy()
        test_df = df_clean[df_clean['is_train'] == 0].copy()
        
        # 只对训练集进行异常值处理
        if target_col in train_df.columns:
            # 对目标列使用 IQR
            Q1 = train_df[target_col].quantile(0.25)
            Q3 = train_df[target_col].quantile(0.75)
            IQR = Q3 - Q1
            
            if IQR > 0 and not pd.isna(IQR):
                lower = Q1 - 3 * IQR
                upper = Q3 + 3 * IQR
                before_rows = len(train_df)
                train_df = train_df[(train_df[target_col] >= lower) & (train_df[target_col] <= upper)].copy()
                after_rows = len(train_df)
                print(f"✅ 异常值处理完成（基于目标列）：移除 {before_rows - after_rows} 行异常值")
        
        # 重新组合数据
        df_clean = pd.concat([train_df, test_df], axis=0)
    
        # 恢复 is_train 列
        if is_train_col is not None:
            df_clean['is_train'] = is_train_col.reindex(df_clean.index).fillna(0).astype(int)
    
        return df_clean
    
    def prepare_modeling_data(self, combined_df, target_col='Price'):
        """准备建模数据 - 修复版本"""
        # 分离训练集和测试集
        train_df = combined_df[combined_df['is_train'] == 1].copy()
        test_df = combined_df[combined_df['is_train'] == 0].copy()
        
        print(f"准备建模数据 - 训练集: {train_df.shape}, 测试集: {test_df.shape}")
        
        # 移除标识列
        train_df = train_df.drop('is_train', axis=1, errors='ignore')
        test_df = test_df.drop('is_train', axis=1, errors='ignore')
        
        # 处理目标变量 - 对数转换（只对训练集）
        if target_col in train_df.columns:
            # 确保目标变量没有NaN
            train_df = train_df.dropna(subset=[target_col])
            y_train = np.log1p(train_df[target_col])
            train_df = train_df.drop([target_col], axis=1)  # 移除原目标列
        else:
            y_train = None
        
        # 移除测试集中的目标列（如果存在）
        if target_col in test_df.columns:
            test_df = test_df.drop(target_col, axis=1)
        
        # 选择数值型特征
        train_numeric_cols = train_df.select_dtypes(include=[np.number]).columns
        test_numeric_cols = test_df.select_dtypes(include=[np.number]).columns
        common_numeric_cols = train_numeric_cols.intersection(test_numeric_cols)
        
        X_train = train_df[common_numeric_cols]
        X_test = test_df[common_numeric_cols]
        
        # 最终检查并处理NaN
        if X_train.isnull().any().any():
            print("警告: 训练集仍存在NaN，进行最终清理...")
            # 移除仍有NaN的列
            nan_cols = X_train.columns[X_train.isnull().any()].tolist()
            X_train = X_train.drop(columns=nan_cols)
            X_test = X_test.drop(columns=nan_cols)
            print(f"移除仍有NaN的列: {nan_cols}")
        
        # 确保没有NaN
        X_train = X_train.fillna(0)
        X_test = X_test.fillna(0)
        
        print(f"建模数据: X_train {X_train.shape}, X_test {X_test.shape}")
        print(f"特征数量: {X_train.shape[1]}")
        
        return X_train, X_test, y_train

class EnhancedLinearModel:
    """改进的线性模型类 - 基于Midterm_codes思路"""
    
    def __init__(self, property_type='price'):
        self.property_type = property_type
        self.models = {}
        self.best_model = None
        self.best_model_name = None
        
    def create_optimized_elasticnet_params(self):
        """优化ElasticNet参数网格，减少训练时间"""
        return {
            'model__alpha': [0.001, 0.01, 0.1],
            'model__l1_ratio': [0.3, 0.5, 0.7],
            'model__max_iter': [1000],
            'model__tol': [1e-3]
        }
    
    def create_mae_scorer(self):
        """创建MAE评分器（在原始价格尺度上）"""
        def original_price_mae_scorer(y_log, y_pred_log):
            y_orig = np.expm1(y_log)
            y_pred_orig = np.expm1(y_pred_log)
            y_pred_orig = np.nan_to_num(y_pred_orig, nan=0.0, posinf=np.finfo(np.float64).max, neginf=0.0)
            y_pred_orig = np.clip(y_pred_orig, 0, None)
            return mean_absolute_error(y_orig, y_pred_orig)
        
        return make_scorer(original_price_mae_scorer, greater_is_better=False)
    
    def create_model_pipeline(self, model):
        """创建包含数据预处理的模型管道"""
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])
        return pipeline
    
    def train_models(self, X_train, y_train, X_val=None, y_val=None):
        """训练多个线性模型"""
        results = []
        mae_scorer = self.create_mae_scorer()
        
        # 定义模型和参数网格
        models_config = {
            'OLS': {
                'model': LinearRegression(),
                'params': {}
            },
            'LASSO': {
                'model': Lasso(random_state=111, max_iter=5000),
                'params': {'model__alpha': [0.001, 0.01, 0.1, 1.0]}
            },
            'Ridge': {
                'model': Ridge(random_state=111),
                'params': {'model__alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}
            },
            'ElasticNet': {
                'model': ElasticNet(random_state=111),
                'params': self.create_optimized_elasticnet_params()
            }
        }
        
        for name, config in models_config.items():
            print(f"\n=== 训练模型: {name} ===")
            
            try:
                # 创建管道
                pipeline = self.create_model_pipeline(config['model'])
                
                if config['params']:
                    # 使用GridSearchCV调参
                    grid = GridSearchCV(
                        pipeline, 
                        config['params'], 
                        cv=min(5, max(2, len(X_train)//10)),
                        scoring=mae_scorer,
                        n_jobs=-1,
                        verbose=0
                    )
                    grid.fit(X_train, y_train)
                    best_model = grid.best_estimator_
                    best_params = grid.best_params_
                else:
                    # 直接训练
                    best_model = pipeline
                    best_model.fit(X_train, y_train)
                    best_params = {}
                
                # 存储模型
                self.models[name] = best_model
                
                # 计算指标
                y_train_pred = best_model.predict(X_train)
                mae_train = mean_absolute_error(np.expm1(y_train), np.expm1(y_train_pred))
                
                if X_val is not None and y_val is not None:
                    y_val_pred = best_model.predict(X_val)
                    mae_val = mean_absolute_error(np.expm1(y_val), np.expm1(y_val_pred))
                else:
                    mae_val = mae_train
                
                # 交叉验证
                cv_folds = min(6, max(2, len(X_train)//5))
                cv_scores = cross_validate(
                    best_model, X_train, y_train, 
                    cv=cv_folds, 
                    scoring=mae_scorer,
                    n_jobs=-1
                )
                mae_cv = -np.mean(cv_scores['test_score'])
                
                # 估算Kaggle分数
                if self.property_type == 'price':
                    kaggle_score = max(0, 100 - (mae_val / 50000))
                else:
                    kaggle_score = max(0, 100 - (mae_val / 500))
                
                result = {
                    'Model': name,
                    'In_sample_MAE': mae_train,
                    'Out_sample_MAE': mae_val,
                    'CV_MAE': mae_cv,
                    'Kaggle_Score': kaggle_score,
                    'Best_Params': best_params
                }
                results.append(result)
                
                print(f"{name} 完成: In-sample MAE = {mae_train:,.2f}, Out-sample MAE = {mae_val:,.2f}")
                
            except Exception as e:
                print(f"{name} 训练失败: {e}")
                continue
        
        # 选择最佳模型
        if results:
            results_df = pd.DataFrame(results)
            best_idx = results_df['Out_sample_MAE'].idxmin()
            self.best_model_name = results_df.loc[best_idx, 'Model']
            self.best_model = self.models.get(self.best_model_name)
            
            print(f"\n最佳模型: {self.best_model_name}")
            print(results_df.to_string(index=False))
            
            return results_df
        else:
            print("所有模型训练失败")
            return pd.DataFrame()
    
    def predict(self, X):
        """使用最佳模型进行预测"""
        if self.best_model is None:
            raise ValueError("没有可用的训练模型")
        
        y_pred_log = self.best_model.predict(X)
        y_pred = np.expm1(y_pred_log)
        
        # 处理负值和异常值
        y_pred = np.clip(y_pred, 0, None)
        y_pred = np.nan_to_num(y_pred, nan=np.median(y_pred))
        
        return y_pred

def main():
    print("=== 改进的房地产价格预测系统 ===\n")
    
    # 文件路径 - 请确保这些文件存在
    price_train_path = "ruc_Class25Q2_train_price.csv"
    price_test_path = "ruc_Class25Q2_test_price.csv"
    rent_train_path = "ruc_Class25Q2_train_rent.csv" 
    rent_test_path = "ruc_Class25Q2_test_rent.csv"
    
    processor = AdvancedRealEstateProcessor()
    all_predictions = []
    presentation_results = {}
    
    # ===== 房价预测 =====
    print("\n" + "="*50)
    print("房价建模")
    print("="*50)
    
    combined_price, price_train, price_test = processor.load_and_combine_data(
        price_train_path, price_test_path, 'Price'
    )
    
    if combined_price is not None:
        # 数据清洗和特征工程
        combined_price = processor.remove_data_leakage(combined_price)
        combined_price_feat = processor.create_advanced_features(combined_price)
        combined_price_clean = processor.handle_missing_values(combined_price_feat, 'Price')
        combined_price_clean = processor.remove_outliers(combined_price_clean, 'Price')
        
        # 准备建模数据
        X_price_train, X_price_test, y_price_train = processor.prepare_modeling_data(
            combined_price_clean, 'Price'
        )
        
        if X_price_train is not None and y_price_train is not None and len(X_price_train) > 0:
            # 分割训练验证集
            X_tr, X_val, y_tr, y_val = train_test_split(
                X_price_train, y_price_train, test_size=0.2, random_state=111
            )
            
            # 训练模型
            price_predictor = EnhancedLinearModel('price')
            price_results = price_predictor.train_models(X_tr, y_tr, X_val, y_val)
            presentation_results['Price'] = price_results
            
            # 预测测试集
            if X_price_test is not None and not X_price_test.empty:
                price_preds = price_predictor.predict(X_price_test)
                test_ids = price_test['ID'] if 'ID' in price_test.columns else range(len(price_preds))
                
                for idx, pred in zip(test_ids, price_preds):
                    all_predictions.append({'ID': idx, 'Price': pred})
                
                print(f"房价测试集预测完成: {len(price_preds)} 条记录")
        else:
            print("房价数据准备失败，跳过房价建模")
    
    # ===== 房租预测 =====  
    print("\n" + "="*50)
    print("房租建模")
    print("="*50)
    
    combined_rent, rent_train, rent_test = processor.load_and_combine_data(
        rent_train_path, rent_test_path, 'Price'
    )
    
    if combined_rent is not None:
        # 数据清洗和特征工程
        combined_rent = processor.remove_data_leakage(combined_rent)
        combined_rent_feat = processor.create_advanced_features(combined_rent)
        combined_rent_clean = processor.handle_missing_values(combined_rent_feat, 'Price')
        combined_rent_clean = processor.remove_outliers(combined_rent_clean, 'Price')
        
        # 准备建模数据
        X_rent_train, X_rent_test, y_rent_train = processor.prepare_modeling_data(
            combined_rent_clean, 'Price'
        )
        
        if X_rent_train is not None and y_rent_train is not None and len(X_rent_train) > 0:
            # 分割训练验证集
            X_tr, X_val, y_tr, y_val = train_test_split(
                X_rent_train, y_rent_train, test_size=0.2, random_state=111
            )
            
            # 训练模型
            rent_predictor = EnhancedLinearModel('rent')
            rent_results = rent_predictor.train_models(X_tr, y_tr, X_val, y_val)
            presentation_results['Rent'] = rent_results
            
            # 预测测试集
            if X_rent_test is not None and not X_rent_test.empty:
                rent_preds = rent_predictor.predict(X_rent_test)
                test_ids = rent_test['ID'] if 'ID' in rent_test.columns else range(len(rent_preds))
                
                for idx, pred in zip(test_ids, rent_preds):
                    all_predictions.append({'ID': idx, 'Price': pred})
                
                print(f"房租测试集预测完成: {len(rent_preds)} 条记录")
        else:
            print("房租数据准备失败，跳过房租建模")
    
    # ===== 输出结果 =====
    if all_predictions:
        # 保存预测结果
        submission_df = pd.DataFrame(all_predictions)
        submission_df.to_csv('submission_Class25Q2.csv', index=False)
        print(f"\n预测结果已保存: submission_Class25Q2.csv ({len(submission_df)} 条记录)")
        
        # 保存展示表格
        for prop_type, results in presentation_results.items():
            if not results.empty:
                # 格式化展示表格
                display_cols = ['Model', 'In_sample_MAE', 'Out_sample_MAE', 'CV_MAE', 'Kaggle_Score']
                display_df = results[display_cols].copy()
                display_df.columns = ['Model', 'In sample', 'Out of sample', 'Cross-validation', 'Kaggle Score']
                
                # 格式化数字
                for col in ['In sample', 'Out of sample', 'Cross-validation']:
                    display_df[col] = display_df[col].apply(lambda x: f"{x:,.2f}")
                display_df['Kaggle Score'] = display_df['Kaggle Score'].apply(lambda x: f"{x:.1f}")
                
                print(f"\n{prop_type} 模型结果:")
                print(display_df.to_string(index=False))
                
                # 保存为CSV
                results.to_csv(f'presentation_results_{prop_type}.csv', index=False)
    
    else:
        print("\n未生成任何预测结果")

if __name__ == "__main__":
    main()

=== 改进的房地产价格预测系统 ===


房价建模
数据加载成功: 训练集 (103871, 55), 测试集 (34017, 55)
移除高缺失率特征: ['抵押信息', 'ID', '套内面积_数值', '套内面积_对数']
已用中位数填充数值列 区县：填充 10973 个缺失值（中位数=68.0）
已用中位数填充数值列 板块_comm：填充 11390 个缺失值（中位数=585.0）
已用中位数填充数值列 容 积 率：填充 42457 个缺失值（中位数=2.5）
已用中位数填充数值列 停车位：填充 44139 个缺失值（中位数=712.0）
已用中位数填充数值列 coord_x：填充 10846 个缺失值（中位数=117.27406399467887）
已用中位数填充数值列 coord_y：填充 10846 个缺失值（中位数=32.21977660956047）
已用中位数填充数值列 房间数：填充 1463 个缺失值（中位数=3.0）
已用中位数填充数值列 客厅数：填充 1463 个缺失值（中位数=2.0）
已用中位数填充数值列 卫生间数：填充 594 个缺失值（中位数=1.0）
已用中位数填充数值列 建筑年份：填充 44507 个缺失值（中位数=2008.0）
已用中位数填充数值列 房龄：填充 44507 个缺失值（中位数=17.0）
已用中位数填充数值列 电梯数：填充 3254 个缺失值（中位数=2.0）
已用中位数填充数值列 每层户数：填充 3254 个缺失值（中位数=4.0）
已用中位数填充数值列 梯户比：填充 3254 个缺失值（中位数=0.4）
已用中位数填充数值列 交易时间_年份：填充 34017 个缺失值（中位数=2024.0）
已用中位数填充数值列 交易时间_月份：填充 34017 个缺失值（中位数=7.0）
已用中位数填充数值列 上次交易_年份：填充 59483 个缺失值（中位数=2017.0）
已用中位数填充数值列 上次交易_月份：填充 59483 个缺失值（中位数=7.0）
已用中位数填充数值列 持有天数：填充 59483 个缺失值（中位数=2598.0）
✅ 缺失值处理完成，is_train 列已恢复并对齐。
✅ 异常值处理完成（基于目标列）：移除 3270 行异常值
准备建模数据 - 训练集: (100601, 82), 测试集